# UK 02 · 区域与站点表

运行 `uk.regions.derive`, `uk.substations.derive`（要求 `01_download.py` 已完成，否则 fail-closed），随后按 `general.toml` 的 `[display].required_sections` 展示：来源与许可、schema、覆盖统计、单位与时点、标准图。

数据产物写 `data/datasets/2_derived/uk/bplus/`，图写 `results/1_DataOverview/1_UK/figures/`。Notebook 本身不保存任何数据；已完成的单元会 SKIP 并说明原因，`stage.run_step(..., refresh=True)` 强制重做。

In [ ]:
from sglib.core.infra.paths import find_repo_root
from sglib.dataoverview import stage
from sglib.dataoverview.overview import figures, tables
from sglib.dataoverview.reader import canonical_paths, read_artifact
import pandas as pd

ROOT = find_repo_root()
COUNTRY = "uk"
config = stage.country_config(ROOT, COUNTRY)
context = stage.country_context(ROOT, COUNTRY)
FIGURES = stage.figures_root(ROOT, COUNTRY)
LABEL = config.country_profile.label
GROUP_KEY = config.values["regions"].get("group_key")
GROUPS = [str(item["id"]) for item in config.values["regions"]["items"]]
contract = config.values["station_contract"]
SOURCE_KEY = str(contract["source_key"])
REGION_COLUMN = str(contract["region_column"])

report = stage.run_step(ROOT, COUNTRY, ["regions", "substations"])
report

## 1 · 来源与许可（source_and_license）

In [ ]:
tables.dataset_ledger(config.values, categories=["boundaries_grid", "regional_totals", "station_register"])

In [ ]:
tables.product_ledger(config.values, products=["regions", "substations"])

## 2 · Schema

正式 schema 合同在 `../schemas/regions_uk.toml` 与 `../schemas/stations_uk.toml`；下表是落地文件的实际列。

In [ ]:
paths = canonical_paths(ROOT, COUNTRY, configuration=config.values)
regions = read_artifact(paths["regions"])
stations = read_artifact(paths["stations"])
tables.schema_table({"regions": regions, "stations": stations})

## 3 · 覆盖统计（coverage_stats）

区域表是保留区域的全集；站点表包含全部站点，未落入保留区域的站点 region 列为空，标准图只绘制落入区域的站点。

In [ ]:
tables.regions_summary(regions, contract)

In [ ]:
regions.drop(columns="geometry").head(8)

In [ ]:
tables.stations_summary(stations, contract)

In [ ]:
stations.drop(columns="geometry").head(8)

按预设分析区域（`regions.items`）汇总源行数、站点数与需求。

In [ ]:
tables.group_summary(regions, stations, group_key=GROUP_KEY, groups=GROUPS, station_contract=contract)

## 4 · 单位与时点（unit_and_vintage）

In [ ]:
tables.unit_and_vintage(config)

## 5 · 标准图（standard_figures）

In [ ]:
fig = figures.plot_regions_and_stations(regions, stations, title=f"{LABEL}: retained regions and stations", region_column=REGION_COLUMN)
figures.save_figure(fig, FIGURES / "02_regions_stations.png")
fig

In [ ]:
fig = figures.plot_regions_and_stations(
    regions, stations,
    title=f"{LABEL}: {len(GROUPS)} analysis regions ({GROUP_KEY})",
    highlight_column=GROUP_KEY, highlight_values=GROUPS, highlight_label="analysis regions",
    region_column=REGION_COLUMN,
)
figures.save_figure(fig, FIGURES / "02_analysis_regions.png")
fig

In [ ]:
fig = figures.plot_group_zoom(regions, stations, group_column=GROUP_KEY, group_value=GROUPS[0], label_column=SOURCE_KEY, region_column=REGION_COLUMN)
figures.save_figure(fig, FIGURES / f"02_zoom_{GROUPS[0]}.png")
fig

下一步：`03_grid.ipynb`。